In [61]:

import streamlit as st
import pandas as pd
import joblib

# Page configuration 
st.set_page_config(page_title="ObesitySense: Obesity level Analysis and Prediction", page_icon="🍔", layout="wide")

# Title
st.title("🍔 ObesitySense: Obesity level Analysis and Prediction")

# Sidebar navigation
st.sidebar.title("Navigation")
page = st.sidebar.radio(
    "Go to:",
    ("Home - Prediction", "Dashboard", "Model Information")
)

# Loading Model, Scaler, Encoder
model = joblib.load(open('models/RandomForest.pkl', 'rb'))
scaler = joblib.load(open('scaler.pkl', 'rb'))  # If you scaled features
label_encoder = joblib.load(open('encoders/label_encoder_target.pkl', 'rb'))
onehot_encoder = joblib.load(open('encoders/onehot_encoder_MTRANS.pkl', 'rb'))
ordinal_encoder = joblib.load(open('encoders/ordinal_encoders.pkl', 'rb'))
ptransformer = joblib.load(open('transformers/yeojohn_ptransformer.pkl', 'rb'))


In [121]:
ordinal_cols = [ 'FCVC', 'NCP', 'CAEC', 'CH2O', 'FAF', 'TUE', 'CALC', 'family_history_with_overweight', 'FAVC', 'SMOKE', 'SCC', 'Gender']
onehot_cols = ['MTRANS']
label_cols = ["NObeyesdad"]
yeo_john_cols = ["Age"]
scale_cols = ['Age','Height', 'Weight']

def preprocess_user_input(user_input):
    # Scale
    user_input[scale_cols] = scaler.transform(user_input[scale_cols])
    
    # Yeo-Johnson only for Age
    user_input[yeo_john_cols] = ptransformer.transform(user_input[yeo_john_cols])
    
    # Ordinal encode
    for col in ordinal_cols:
        user_input[[col]] = ordinal_encoder[col].transform(user_input[[col]])
    
    # Onehot encode
    onehot_transformed = onehot_encoder.transform(user_input[onehot_cols])
    onehot_df = pd.DataFrame(onehot_transformed, columns=onehot_encoder.get_feature_names_out(onehot_cols))
    
    user_input = user_input.drop(columns=onehot_cols).reset_index(drop=True)
    user_input = pd.concat([user_input, onehot_df], axis=1)

    return user_input


In [107]:
if page == "Home - Prediction":
    st.header("Predict Your Obesity Level 🧮")

    with st.form("prediction_form"):
        st.subheader("Enter Your Details:")

        
        Gender = st.selectbox("Gender", ['Female', 'Male'])
        
        # Continuous Inputs (numerical)
        Age = st.number_input("Age", min_value=1, max_value=100, value=25)
        Height = st.number_input("Height (in meters)", min_value=0.5, max_value=2.5, value=1.70)
        Weight = st.number_input("Weight (in kg)", min_value=10, max_value=300, value=70)

        # Ordinal Features (select box)
        family_history_with_overweight = st.selectbox("Family History with Overweight?", ["no", "yes"])
        FAVC = st.selectbox("Do you eat high calorie food?", ["no", "yes"])
        FCVC = st.selectbox("Frequency of Vegetable Consumption in Meals", ["Never", "Sometimes", "Always"])
        NCP = st.selectbox("Number of Main Meals in a day", ["Btwn 1 & 2","3", "More than 3", "No Answer"])
        CAEC = st.selectbox("Do you eat food between meals?", ['no', 'Sometimes', 'Frequently', "Always"])
        SMOKE = st.selectbox("Do You Smoke?", ["no", "yes"])
        CH2O = st.selectbox("Daily Water Intake", ['less than 1L', 'Btwn 1L & 2L', 'More than 2L'])
        SCC = st.selectbox("Do You Monitor Calories?", ["no", "yes"])
        FAF = st.selectbox("Physical Activity Frequency", ['Never','1 to 2times','2 to 4 times', '4 or 5 times'])
        TUE = st.selectbox("Time Using Technology Devices", ['0 to 2h','3 to 5h', 'More than 5h'])
        CALC = st.selectbox("Alcohol Consumption", ['no', 'Sometimes', 'Frequently', 'Always'])

        # OneHot Encoded Feature
        MTRANS = st.selectbox("Mode of Transportation", ["Automobile", "Bike", "Motorbike", "Public_Transportation", "Walking"])

        submit = st.form_submit_button("Predict")
        if submit:
    # Creating a DataFrame with the user input
            user_input = pd.DataFrame({
                "Gender": [Gender],
                 "Age": [Age],
                 "Height": [Height],
                "Weight": [Weight],
                "family_history_with_overweight": [family_history_with_overweight],
                "FAVC": [FAVC],
                "FCVC": [FCVC],
                "NCP": [NCP],
                "CAEC": [CAEC],
                "SMOKE": [SMOKE],
                "CH2O": [CH2O],
                "SCC": [SCC],
                "FAF": [FAF],
                "TUE": [TUE],
                "CALC": [CALC],
                "MTRANS": [MTRANS]
            })

            # applying the preprocessing on user inputs
            preprocessed_input = preprocess_user_input(user_input)


            # Prediction
            prediction = model.predict(preprocessed_input)

            # Decoding the prediction back to original class names
            prediction_class = label_encoder.inverse_transform(prediction)
    
            st.write(f"The predicted obesity level is: {prediction_class[0]}")



In [4]:
if page == "Dashboard":
    st.title("Dashboard - Data Insights")
    
    # Load the data 
    df_og = pd.read_csv('ObesityDataSet_raw_and_data_sinthetic.csv')
    df_preprocessed = pd.read_csv('preprocessed_obesity.csv')
    
    # Display basic summary statistics
    st.header("Summary Statistics")
    st.write(df_og.describe())




NameError: name 'page' is not defined

In [127]:
import seaborn as sns
import matplotlib.pyplot as plt

if page == "Dashboard":
    st.title("Dashboard - Data Insights")
    
    # Correlation Heatmap
    st.header("Feature Correlation Heatmap")
    correlation_matrix = df_preprocessed[['Age', 'Weight', 'Height', 'FCVC', 'NCP', 'CAEC', 'CH2O', 'FAF', 'TUE', 'CALC', 'FAVC']].corr() 
    
    # Plotting the heatmap
    fig, ax = plt.subplots(figsize=(7, 4))
    sns.heatmap(correlation_matrix, annot=True, cmap="coolwarm", fmt=".2f", ax= ax)
    st.pyplot(fig, use_container_width=True)  


In [117]:
if page == "Dashboard":
    st.title("Dashboard - Data Insights")

    # Display Boxplot for Age by Obesity level 
    st.header("Age Distribution by Obesity Level")
    fig, ax = plt.subplots(figsize=(7, 4))
    sns.boxplot(x="NObeyesdad", y="Age", data=df_og, ax = ax)
    st.pyplot(fig,use_container_width=True)  # Display the plot


In [123]:
from sklearn.decomposition import PCA

if page == "Dashboard":
    st.title("Dashboard - Data Insights")

    # PCA 
    st.header("PCA Visualization")
    
    pca = PCA(n_components=2)
    pca_result = pca.fit_transform(df_preprocessed.drop(columns=["NObeyesdad"])) 
    
    # Plotting the PCA graph
    fig, ax = plt.subplots(figsize=(7, 4))
    plt.scatter(pca_result[:, 0], pca_result[:, 1], c=df_preprocessed["NObeyesdad"], cmap='viridis')
    plt.xlabel("PCA 1")
    plt.ylabel("PCA 2")
    plt.title("PCA of Features by Obesity Level")
    st.pyplot(fig,use_container_width=True)  


In [77]:
if page == "Model Information":
    st.title("Model Information")

    # Model summary
    st.header("Model: Random Forest Classifier")
    st.write("This model was trained using the Random Forest Classifier, which is an ensemble learning method. It works by building multiple decision trees and combining their results to improve accuracy and reduce overfitting.")
    
    
    st.header("Model Evaluation Metrics (Before Optimization)")

    st.write("Accuracy: 94%")  
    st.write("Classification Report:") 

    # Display confusion matrix
    st.write("Confusion Matrix:")
    st.image('confusion_matrix.png') 


In [125]:
if page == "Model Information":
    

    st.header("Model Optimization")
    st.write("I performed hyperparameter tuning using RandomizedSearchCV and it improved the results by 1%")
    st.write("Feature Importance is visualized to show which features contribute the most to the model's prediction.")
    
    # Feature Importance
    st.header("Feature Importance")

    feature_names = ['Gender', 'Age', 'Height', 'Weight', 'family_history_with_overweight',
       'FAVC', 'FCVC', 'NCP', 'CAEC', 'SMOKE', 'CH2O', 'SCC', 'FAF', 'TUE',
       'CALC', 'MTRANS_Automobile', 'MTRANS_Bike', 'MTRANS_Motorbike',
       'MTRANS_Public_Transportation', 'MTRANS_Walking'] 

    importances = model.feature_importances_

    # Create a DataFrame for better plotting
    feat_importance_df = pd.DataFrame({
        'Feature': feature_names,
        'Importance': importances
    }).sort_values(by='Importance', ascending=False)

    # Plot
    fig, ax = plt.subplots(figsize=(7, 4))
    sns.barplot(x='Importance', y='Feature', data=feat_importance_df, palette='viridis', ax = ax)
    plt.title('Feature Importance of Random Forest Model')
    plt.xlabel('Importance Score')
    plt.ylabel('Feature')

    st.pyplot(fig,use_container_width=True)  
